# ETL Logcomex sem SQLite

Este notebook l? os arquivos `.xlsx` diretamente para DataFrames e substitui as consultas SQL por transforma??es em Python/pandas.


In [ ]:
# Bibliotecas
from pathlib import Path

import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


In [ ]:
# Caminhos dos arquivos
input_dir = Path("..") / "Input"
depara_output_dir = Path("..") / "De-Para" / "Output GPT"

ns = {
    "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}
rel_ns = {"rel": "http://schemas.openxmlformats.org/package/2006/relationships"}

TEXT_FORCE_COLUMNS = {
    "ncm",
    "cod_cgce",
    "cod_cnae_importador",
}


In [ ]:
# Fun??es utilit?rias para ler XLSX e criar DataFrames

def col_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    n = 0
    for ch in letters:
        n = n * 26 + ord(ch.upper()) - 64
    return n - 1


def read_xlsx_first_sheet(path):
    path = Path(path)
    with zipfile.ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                parts = [t.text or "" for t in si.findall(".//a:t", ns)]
                shared.append("".join(parts))

        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rid_to_target = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("rel:Relationship", rel_ns)}
        first_sheet = workbook.find("a:sheets", ns).findall("a:sheet", ns)[0]
        rid = first_sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        sheet_path = "xl/" + rid_to_target[rid].lstrip("/")
        sheet = ET.fromstring(z.read(sheet_path))

        rows = []
        max_col = 0
        for row in sheet.findall(".//a:sheetData/a:row", ns):
            values = []
            for cell in row.findall("a:c", ns):
                idx = col_index(cell.attrib.get("r", "A1"))
                while len(values) <= idx:
                    values.append(None)

                cell_type = cell.attrib.get("t")
                value_node = cell.find("a:v", ns)
                value = None if value_node is None else value_node.text

                if cell_type == "s" and value is not None:
                    value = shared[int(value)]
                elif cell_type == "inlineStr":
                    value = "".join(t.text or "" for t in cell.findall(".//a:t", ns))

                values[idx] = value

            max_col = max(max_col, len(values))
            rows.append(values)

    if not rows:
        return [], []

    rows = [row + [None] * (max_col - len(row)) for row in rows]
    headers = [str(col).strip() if col is not None else f"coluna_{i + 1}" for i, col in enumerate(rows[0])]
    return headers, rows[1:]


def normalize_column_name(name):
    normalized = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii")
    normalized = normalized.lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")
    return normalized or "coluna"


def unique_column_names(columns):
    seen = {}
    result = []
    for column in columns:
        base = normalize_column_name(column)
        count = seen.get(base, 0) + 1
        seen[base] = count
        result.append(base if count == 1 else f"{base}_{count}")
    return result


def can_parse_float(value):
    try:
        float(str(value).replace(",", "."))
        return True
    except (TypeError, ValueError):
        return False


def can_parse_int(value):
    try:
        return float(str(value).replace(",", ".")).is_integer()
    except (TypeError, ValueError):
        return False


def coerce_dataframe_types(df, text_force=None):
    text_force = set(text_force or [])
    result = df.copy()

    for column in result.columns:
        non_null = result[column].dropna()
        non_null = non_null[non_null.astype(str).str.strip().ne("")]

        if column in text_force or non_null.empty:
            result[column] = result[column].astype("string")
            continue

        if non_null.map(can_parse_int).all():
            result[column] = pd.to_numeric(result[column].astype("string").str.replace(",", ".", regex=False), errors="coerce").astype("Int64")
        elif non_null.map(can_parse_float).all():
            result[column] = pd.to_numeric(result[column].astype("string").str.replace(",", ".", regex=False), errors="coerce")
        else:
            result[column] = result[column].astype("string")

    return result


def read_xlsx_to_dataframe(path, text_force=None):
    headers, rows = read_xlsx_first_sheet(path)
    if not headers:
        raise ValueError(f"Arquivo sem cabe?alho/dados: {path}")

    columns = unique_column_names(headers)
    df = pd.DataFrame(rows, columns=columns)
    df = df.replace("", pd.NA)
    return coerce_dataframe_types(df, text_force=text_force)


def list_xlsx_files(folder_path):
    folder_path = Path(folder_path)
    files = sorted(path for path in folder_path.glob("*.xlsx") if not path.name.startswith("~$"))
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em: {folder_path}")
    return files


def read_single_xlsx_from_folder(folder_path, text_force=None):
    files = list_xlsx_files(folder_path)
    if len(files) != 1:
        raise RuntimeError(f"Esperado 1 arquivo .xlsx em {folder_path}, encontrados {len(files)}")
    return files[0], read_xlsx_to_dataframe(files[0], text_force=text_force)


def normalize_match_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def normalize_match_series(series):
    return series.astype("string").fillna("").str.strip().str.lower()


In [ ]:
# Leitura direta dos arquivos para DataFrames
input_file, df_ext_logcomex = read_single_xlsx_from_folder(input_dir, text_force=TEXT_FORCE_COLUMNS)

depara_files = {path.stem: path for path in list_xlsx_files(depara_output_dir)}

df_depara_exportador_player = read_xlsx_to_dataframe(depara_files["depara_exportador_player"])
df_depara_paises_exportador = read_xlsx_to_dataframe(depara_files["depara_paises_exportador"])
df_depara_palavrachave_exportador = read_xlsx_to_dataframe(depara_files["depara_palavrachave_exportador"])

df_arquivos_lidos = pd.DataFrame([
    {"dataframe": "df_ext_logcomex", "arquivo": input_file.name, "linhas": len(df_ext_logcomex), "colunas": df_ext_logcomex.shape[1]},
    {"dataframe": "df_depara_exportador_player", "arquivo": depara_files["depara_exportador_player"].name, "linhas": len(df_depara_exportador_player), "colunas": df_depara_exportador_player.shape[1]},
    {"dataframe": "df_depara_paises_exportador", "arquivo": depara_files["depara_paises_exportador"].name, "linhas": len(df_depara_paises_exportador), "colunas": df_depara_paises_exportador.shape[1]},
    {"dataframe": "df_depara_palavrachave_exportador", "arquivo": depara_files["depara_palavrachave_exportador"].name, "linhas": len(df_depara_palavrachave_exportador), "colunas": df_depara_palavrachave_exportador.shape[1]},
])

display(df_arquivos_lidos)
display(df_ext_logcomex.head())


In [ ]:
# DataFrame equivalente ao antigo resultado de depara_exportador_player
# Mant?m vari?veis parecidas com o notebook original, mas sem SQL.

teste = df_depara_exportador_player.copy()

display(teste.head())
teste.shape


In [ ]:
# Mapeamento de exportador e player em Python/pandas

def prepare_keyword_rules(df_palavrachave):
    rules = df_palavrachave[["palavra_chave", "provavel_exportador"]].dropna(subset=["palavra_chave", "provavel_exportador"]).copy()
    rules["palavra_chave_match"] = normalize_match_series(rules["palavra_chave"])
    rules = rules[rules["palavra_chave_match"].ne("")]
    return list(rules[["palavra_chave_match", "palavra_chave", "provavel_exportador"]].itertuples(index=False, name=None))


def find_keyword_exporter(description, keyword_rules):
    description_match = normalize_match_value(description)
    if not description_match:
        return pd.Series({"exportador_palavra_chave": pd.NA, "palavra_chave_mapeada": pd.NA})

    for keyword_match, keyword_original, exporter in keyword_rules:
        if keyword_match and keyword_match in description_match:
            return pd.Series({"exportador_palavra_chave": exporter, "palavra_chave_mapeada": keyword_original})

    return pd.Series({"exportador_palavra_chave": pd.NA, "palavra_chave_mapeada": pd.NA})


def build_country_exporter_map(df_paises):
    df = df_paises[["pais_de_aquisicao", "pais_de_origem", "provavel_exportador"]].dropna(subset=[
        "pais_de_aquisicao",
        "pais_de_origem",
        "provavel_exportador",
    ]).copy()
    df["pais_de_aquisicao_match"] = normalize_match_series(df["pais_de_aquisicao"])
    df["pais_de_origem_match"] = normalize_match_series(df["pais_de_origem"])
    df = df[df["pais_de_aquisicao_match"].ne("") & df["pais_de_origem_match"].ne("")]
    df = df.drop_duplicates(["pais_de_origem_match", "pais_de_aquisicao_match"], keep="first")
    return {
        (row.pais_de_origem_match, row.pais_de_aquisicao_match): row.provavel_exportador
        for row in df.itertuples(index=False)
    }


def build_player_map(df_player):
    df = df_player[["provavel_exportador", "player"]].dropna(subset=["provavel_exportador"]).copy()
    df["exportador_match"] = normalize_match_series(df["provavel_exportador"])
    df = df[df["exportador_match"].ne("")]
    df = df.drop_duplicates("exportador_match", keep="first")
    return dict(zip(df["exportador_match"], df["player"]))


def map_exportador_e_player(df_ext, df_palavrachave, df_paises, df_player):
    result = df_ext.copy()

    keyword_rules = prepare_keyword_rules(df_palavrachave)
    keyword_result = result["descricao_produto"].apply(lambda value: find_keyword_exporter(value, keyword_rules))
    result = pd.concat([result, keyword_result], axis=1)

    country_exporter_map = build_country_exporter_map(df_paises)
    country_keys = list(zip(
        normalize_match_series(result["pais_de_origem"]),
        normalize_match_series(result["pais_de_aquisicao"]),
    ))
    result["exportador_pais"] = [country_exporter_map.get(key, pd.NA) for key in country_keys]

    existing_exporter_mask = result["provavel_exportador"].notna() & normalize_match_series(result["provavel_exportador"]).ne("")
    keyword_mask = result["exportador_palavra_chave"].notna()
    country_mask = result["exportador_pais"].notna()

    result["exportador_mapeado"] = pd.NA
    result["regra_utilizada"] = "nenhuma"

    result.loc[country_mask, "exportador_mapeado"] = result.loc[country_mask, "exportador_pais"]
    result.loc[country_mask, "regra_utilizada"] = "pais"

    result.loc[keyword_mask, "exportador_mapeado"] = result.loc[keyword_mask, "exportador_palavra_chave"]
    result.loc[keyword_mask, "regra_utilizada"] = "palavra_chave"

    result.loc[existing_exporter_mask, "exportador_mapeado"] = result.loc[existing_exporter_mask, "provavel_exportador"]
    result.loc[existing_exporter_mask, "regra_utilizada"] = "registro_existente"

    player_map = build_player_map(df_player)
    result["player"] = normalize_match_series(result["exportador_mapeado"]).map(player_map)

    return result


teste1 = map_exportador_e_player(
    df_ext_logcomex,
    df_depara_palavrachave_exportador,
    df_depara_paises_exportador,
    df_depara_exportador_player,
)

display(teste1.head(30))
teste1.shape


In [ ]:
# Resumos de valida??o em pandas

df_resumo_regras = (
    teste1["regra_utilizada"]
    .value_counts(dropna=False)
    .rename_axis("regra_utilizada")
    .reset_index(name="linhas")
)

df_resumo_player = pd.DataFrame({
    "linhas_total": [len(teste1)],
    "exportador_mapeado_preenchido": [int(teste1["exportador_mapeado"].notna().sum())],
    "player_preenchido": [int(teste1["player"].notna().sum())],
    "player_nulo": [int(teste1["player"].isna().sum())],
})

display(df_resumo_regras)
display(df_resumo_player)
